# 九曜：天墟 EP01 — 配音試作（Qwen3-TTS，Kaggle 免費 GPU）

- 模型：**Qwen3-TTS**（Apache 2.0，可商用）。先用 **VoiceDesign** 依角色音色提示詞「設計」出每個角色的聲音，
  再用 **Base** 把那段聲音當參考、唸完該角色所有台詞 → 同一角色每句聲音一致。
- 台詞與音色提示詞已內嵌（來源：repo 的 `production/EP01/shuohao/voice/lines.json`，14 句、6 個聲音：旁白＋5 角色）。
- **Kaggle 設定**：Accelerator = GPU T4 x2、Internet = On。要離開電腦就用 Save Version → Save & Run All。
- 成品：`/kaggle/working/EP01_voice.zip`（`clone/` 每句一檔＝主要版本；`design/` 每句直接設計的對照版；`ref/` 各角色參考聲音）。下載後交給 Claude。

In [ ]:
# 1. 安裝（約 1～2 分鐘）
!pip install -q -U qwen-tts soundfile
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 台詞與音色設定（內容＝repo 的 production/EP01/shuohao/voice/lines.json，改台詞改那份再重產本筆記本）
import json, os
DATA = json.loads(r"""{"lines": [{"id": "E01-01_c2_1", "speaker": "VO", "name": "旁白", "text": "九曜界，天玄院。", "delivery": "旁白：中性成熟、慢；動畫補充的地名旁白，非遊戲台詞", "start": 3, "cutDur": 2}, {"id": "E01-03_c2_1", "speaker": "C02", "name": "齊衡烈", "text": "你就是新來的？", "delivery": "爽朗、直接，像認出新隊友；不是搞笑腔", "start": 15, "cutDur": 3}, {"id": "E01-03_c2_2", "speaker": "C01", "name": "玩家", "text": "應該是。", "delivery": "有點不確定，但語氣平靜", "start": 15, "cutDur": 3}, {"id": "E01-03_c3_1", "speaker": "C02", "name": "齊衡烈", "text": "那就對了。第七室最近正缺個能打的。", "delivery": "笑著拍了一下自己的手臂——是興奮，不是挑釁", "start": 18, "cutDur": 4}, {"id": "E01-04_c2_1", "speaker": "C03", "name": "郁岑燁", "text": "你看都沒看他出手，怎麼知道他能打？", "delivery": "平淡，不是故意嗆人，只是對不精確的判斷沒耐心", "start": 24, "cutDur": 4}, {"id": "E01-04_c3_1", "speaker": "C02", "name": "齊衡烈", "text": "所以才要打一場啊。", "delivery": "理所當然，躍躍欲試", "start": 28, "cutDur": 5}, {"id": "E01-04_c4_1", "speaker": "C03", "name": "郁岑燁", "text": "……無聊。", "delivery": "低頭繼續擦劍，眉頭輕皺", "start": 33, "cutDur": 2}, {"id": "E01-05_c3_1", "speaker": "C04", "name": "厲若楓", "text": "別急。他的腳步有點亂。", "delivery": "平靜，開口前先停一拍；句與句之間有停頓", "start": 39, "cutDur": 3}, {"id": "E01-05_c5_1", "speaker": "C01", "name": "玩家", "text": "你們都看得出來？", "delivery": "真心發問，不是頂嘴", "start": 44, "cutDur": 2}, {"id": "E01-06_c1_1", "speaker": "C03", "name": "郁岑燁", "text": "至少我看得出來。", "delivery": "頭也沒抬，理所當然", "start": 46, "cutDur": 2}, {"id": "E01-06_c2_1", "speaker": "C02", "name": "齊衡烈", "text": "哈哈，那更好！明天演武場見。", "delivery": "大笑，把重刀扛上肩", "start": 48, "cutDur": 4}, {"id": "E01-07_c1_1", "speaker": "C05", "name": "蕭曜霖", "text": "不用等明天。", "delivery": "畫外音，人還沒現身；低沉平穩，不提高音量——環境聲先壓下來", "start": 52, "cutDur": 2}, {"id": "E01-07_c4_1", "speaker": "C05", "name": "蕭曜霖", "text": "半刻鐘後，新生演武場集合。", "delivery": "慢，一字一句；說完不再補充", "start": 58.5, "cutDur": 3}, {"id": "E01-08_c4_1", "speaker": "C01", "name": "玩家", "text": "知道了。", "delivery": "短、乾脆，壓住一點緊張", "start": 70.5, "cutDur": 4}], "voices": {"VO": {"name": "旁白", "prompt": "Mature narrator voice of indeterminate age around forty, neutral warm baritone, low-mid pitch, smooth chest resonance, narrow dynamic range. Moderate-low volume, slow pace, long even phrasing with falling endings. Standard Mandarin with neutral Taiwanese pronunciation. Calm, grounded, unhurried."}, "C01": {"name": "玩家", "prompt": "20-year-old male, clear natural light baritone, mid pitch, moderate chest support, narrow dynamic range. Moderate volume, medium pace, brief pause before answering, plain level phrasing. Standard Mandarin with neutral Taiwanese pronunciation. Calm, slightly unsure."}, "C02": {"name": "齊衡烈", "prompt": "20-year-old male, bright resonant baritone, mid-to-high pitch, strong chest support, wide dynamic range. Loud volume, fast pace, punchy phrase openings, open easy breath. Standard Mandarin with neutral Taiwanese pronunciation. Warm, eager, confident."}, "C03": {"name": "郁岑燁", "prompt": "20-year-old male, clean cool tenor, little breath, mid pitch, firm light support, narrow dynamic range. Low-to-moderate volume, medium-slow pace, clipped short phrases, crisp consonants, falling endings. Standard Mandarin with neutral Taiwanese pronunciation. Reserved, faintly impatient."}, "C04": {"name": "厲若楓", "prompt": "20-year-old male, low soft baritone with a hint of breath, low-mid pitch, relaxed chest resonance, very narrow dynamic range. Quiet volume, slow pace, clear pauses between phrases, even flat intonation. Standard Mandarin with neutral Taiwanese pronunciation. Calm and attentive."}, "C05": {"name": "蕭曜霖", "prompt": "42-year-old male, deep dense bass-baritone, low pitch, heavy chest resonance, firm steady support, narrow controlled dynamic range. Moderate volume that carries, slow deliberate pace, level falling intonation. Standard Mandarin with neutral Taiwanese pronunciation. Stern, steady authority."}}}""")
LINES, VOICES = DATA["lines"], DATA["voices"]
OUT = "/kaggle/working/voice"
for d in ("ref", "clone", "design"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
# 參考句：每個角色先用同一句中性台詞設計聲音（不是劇本台詞，約 5 秒）
REF_TEXT = "今天天氣不錯，我們慢慢走過去吧，路上還可以再聊一聊。"
print(len(LINES), "句；聲音：", {k: v["name"] for k, v in VOICES.items()})

In [ ]:
# 3. 設計每個角色的參考聲音＋每句直接設計版（VoiceDesign 模型）
#    T4 不支援 bf16、也不能用 flash-attn → float32＋sdpa（1.7B 約 7GB 顯存，夠用）
import torch, soundfile as sf, gc
from qwen_tts import Qwen3TTSModel
DTYPE = torch.float32
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=DTYPE, attn_implementation="sdpa")
REFS = {}
for vid, v in VOICES.items():
    wavs, sr = design.generate_voice_design(text=REF_TEXT, language="Chinese", instruct=v["prompt"])
    sf.write(f"{OUT}/ref/{vid}_{v['name']}.wav", wavs[0], sr); REFS[vid] = (wavs[0], sr)
    print("ref", vid, v["name"], f"{len(wavs[0])/sr:.1f}s")
for l in LINES:  # 對照版：每句直接用「音色＋該句語氣」設計（語氣較準，但同一角色每句聲音可能不同）
    ins = VOICES[l["speaker"]]["prompt"] + (" Delivery for this line (Chinese note): " + l["delivery"] if l["delivery"] else "")
    wavs, sr = design.generate_voice_design(text=l["text"], language="Chinese", instruct=ins)
    sf.write(f"{OUT}/design/{l['id']}_{l['name']}.wav", wavs[0], sr)
del design; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 4. 主要版本：用參考聲音複製，唸該角色全部台詞（Base 模型）→ 同角色每句聲音一致
clone = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=DTYPE, attn_implementation="sdpa")
PROMPTS = {vid: clone.create_voice_clone_prompt(ref_audio=REFS[vid], ref_text=REF_TEXT) for vid in VOICES}
REPORT = []
for l in LINES:
    wavs, sr = clone.generate_voice_clone(text=l["text"], language="Chinese", voice_clone_prompt=PROMPTS[l["speaker"]])
    sf.write(f"{OUT}/clone/{l['id']}_{l['name']}.wav", wavs[0], sr)
    sec = len(wavs[0]) / sr
    REPORT.append(f"{l['id']}  {l['name']}  {sec:.2f}s / 分鏡 {l['cutDur']}s  {'⚠️ 超過分鏡秒數' if sec > l['cutDur'] else ''}  {l['text']}")
print("\n".join(REPORT))

In [ ]:
# 5. 打包下載（右側 Output → EP01_voice.zip）
import shutil
open(f"{OUT}/report.txt", "w").write("\n".join(REPORT))
shutil.make_archive("/kaggle/working/EP01_voice", "zip", OUT)
print("OK → /kaggle/working/EP01_voice.zip")

### 常見狀況
- **聲音糊、雜音或破音**：第 3 格 `DTYPE` 已是 float32；若還是有問題截圖給 Claude。
- **`attn_implementation` 錯誤**：把 `"sdpa"` 改成 `"eager"` 重跑第 3、4 格。
- **口音偏大陸腔**：音色提示詞已寫 Taiwanese pronunciation；聽完告訴 Claude 哪個角色要調，只重跑那個角色。